# ROGII: Private Safe Particle + Anchor Submission

This is the deployment notebook for an independently implemented, leakage-safe particle tracker. It follows the hidden-test contract: the supplied visible `TVT_input` prefix is used to initialize the structural state, the suffix `GR` observations update particles against the paired typewell curve, and the posterior mean is blended with the exact last visible TVT anchor.

The deployment constants were frozen before leaderboard evaluation. The original complete 773-well true-boundary run selected a fixed particle weight of **0.625**: its pooled RMSE was 13.112 versus 15.910 for the anchor, and five-fold out-of-well weight fitting gave 13.154. A later audit found that the first seed check overlapped seven of eight particle seeds; a genuinely disjoint post-submit run scored 13.350 with an OOF value of 13.358. This corrected note does not change inference. The notebook does not inspect labels, public predictions, or leaderboard feedback, and it does not tune per test well.

The final cell joins predictions to the official sample by exact row ID and asserts the schema, order, uniqueness, row count, and finiteness before writing `submission.csv`.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data directory was not found')
TEST_DIR = DATA_ROOT / 'test'
SAMPLE_PATH = DATA_ROOT / 'sample_submission.csv'
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Frozen from local true-boundary validation, not leaderboard search.
N_PARTICLES = 300
N_SEEDS = 8
SEED_TEMPERATURE = 20.0
PARTICLE_WEIGHT = 0.625
GLOBAL_SEED = 20260719

test_paths = sorted(TEST_DIR.glob('*__horizontal_well.csv'))
print('Data root:', DATA_ROOT)
print('Test wells:', len(test_paths))
print('Frozen configuration:', {
    'particles': N_PARTICLES,
    'seeds': N_SEEDS,
    'temperature': SEED_TEMPERATURE,
    'particle_weight': PARTICLE_WEIGHT,
})

## State model and prefix-only calibration

Each particle carries `U = TVT + Z` and a slow structural rate `dU/dMD`. At every suffix row the known well trajectory `Z` converts the structural state back to a candidate TVT. The paired typewell GR curve supplies a likelihood after a robust affine GR calibration fitted on visible prefix pairs only. Systematic resampling is triggered when effective particle count falls below half the population.

Eight independent particle paths are combined with tempered sequence evidence. The temperature prevents one stochastic path from taking essentially all posterior mass on long wells. The 37.5% anchor component is a safety prior for repeated or ambiguous GR motifs.

In [ ]:
def split_boundary(frame):
    observed = frame['TVT_input'].notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 2 or observed[missing[0]:].any():
        raise ValueError('Expected a visible TVT prefix and one contiguous missing suffix')
    return int(missing[0])

def calibrate_gr(expected, observed):
    mask = np.isfinite(expected) & np.isfinite(observed)
    x, y = expected[mask], observed[mask]
    if len(x) < 20:
        return 1.0, 0.0, 30.0
    keep = np.ones(len(x), dtype=bool)
    gain, offset = 1.0, 0.0
    for _ in range(3):
        design = np.column_stack((x[keep], np.ones(int(keep.sum()))))
        gain, offset = np.linalg.lstsq(design, y[keep], rcond=None)[0]
        gain = float(np.clip(gain, 0.35, 2.5))
        offset = float(np.median(y[keep] - gain * x[keep]))
        residual = y - (gain * x + offset)
        center = float(np.median(residual[keep]))
        mad = float(1.4826 * np.median(np.abs(residual[keep] - center)))
        if mad <= 1e-8:
            break
        new_keep = np.abs(residual - center) <= 3.5 * mad
        if new_keep.sum() < 20 or np.array_equal(new_keep, keep):
            break
        keep = new_keep
    residual = y - (gain * x + offset)
    sigma = float(1.4826 * np.median(np.abs(residual - np.median(residual))))
    return gain, offset, float(np.clip(sigma, 10.0, 60.0))

def initial_rate(md, z, tvt, window=30):
    md, z, tvt = md[-window:], z[-window:], tvt[-window:]
    delta_md = np.diff(md)
    valid = delta_md > 0
    if valid.sum() < 3:
        return 0.0
    rates = np.diff(tvt + z)[valid] / delta_md[valid]
    return float(np.clip(np.median(rates), -0.25, 0.25))

def systematic_resample(weights, rng):
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    points = rng.uniform(0.0, 1.0 / len(weights)) + np.arange(len(weights)) / len(weights)
    return np.searchsorted(cumulative, points, side='left')

In [ ]:
def run_seed(seed, md, z, gr, previous_md, anchor_u, rate0, tw_tvt, expected_hgr, gr_sigma):
    rng = np.random.default_rng(seed)
    u = anchor_u + 4.5 * rng.standard_normal(N_PARTICLES)
    rate = rate0 + 0.01 * rng.standard_normal(N_PARTICLES)
    weights = np.full(N_PARTICLES, 1.0 / N_PARTICLES)
    prediction = np.empty(len(md))
    log_evidence = 0.0

    for row in range(len(md)):
        delta_md = max(float(md[row] - previous_md), 1.0)
        rate = 0.998 * rate + 0.002 * rng.standard_normal(N_PARTICLES)
        u = u + rate * delta_md + 0.005 * rng.standard_normal(N_PARTICLES)
        particle_tvt = np.clip(u - z[row], tw_tvt[0] - 100.0, tw_tvt[-1] + 100.0)
        u = particle_tvt + z[row]

        if np.isfinite(gr[row]):
            expected = np.interp(particle_tvt, tw_tvt, expected_hgr)
            scaled = (gr[row] - expected) / gr_sigma
            likelihood = np.exp(np.clip(-0.5 * scaled * scaled, -50.0, 0.0))
            evidence = float(weights @ likelihood)
            log_evidence += np.log(max(evidence, 1e-300))
            weights *= likelihood
            total = float(weights.sum())
            weights = weights / total if total > 1e-300 else np.full(N_PARTICLES, 1.0 / N_PARTICLES)

        effective = 1.0 / float(weights @ weights)
        if effective < 0.5 * N_PARTICLES:
            selected = systematic_resample(weights, rng)
            u = u[selected] + 0.10 * rng.standard_normal(N_PARTICLES)
            rate = rate[selected] + 0.001 * rng.standard_normal(N_PARTICLES)
            weights.fill(1.0 / N_PARTICLES)
        prediction[row] = float(weights @ (u - z[row]))
        previous_md = md[row]
    return prediction, log_evidence

def particle_ensemble(frame, typewell, seed_base):
    boundary = split_boundary(frame)
    md = frame['MD'].to_numpy(dtype=float)
    z = frame['Z'].to_numpy(dtype=float)
    gr = frame['GR'].to_numpy(dtype=float)
    known = frame['TVT_input'].to_numpy(dtype=float)[:boundary]
    tw = typewell[['TVT', 'GR']].dropna().sort_values('TVT').drop_duplicates('TVT')
    tw_tvt = tw['TVT'].to_numpy(dtype=float)
    tw_gr = tw['GR'].to_numpy(dtype=float)
    if len(tw_tvt) < 3:
        raise ValueError('Typewell does not contain enough finite rows')

    prefix_expected = np.interp(known, tw_tvt, tw_gr)
    gain, offset, gr_sigma = calibrate_gr(prefix_expected, gr[:boundary])
    expected_hgr = gain * tw_gr + offset
    rate0 = initial_rate(md[:boundary], z[:boundary], known)
    anchor_u = float(known[-1] + z[boundary - 1])

    paths, evidence = [], []
    for seed in range(seed_base, seed_base + N_SEEDS):
        prediction, log_ev = run_seed(
            seed, md[boundary:], z[boundary:], gr[boundary:],
            float(md[boundary - 1]), anchor_u, rate0,
            tw_tvt, expected_hgr, gr_sigma,
        )
        paths.append(prediction)
        evidence.append(log_ev)

    evidence = np.asarray(evidence, dtype=float)
    logits = (evidence - evidence.max()) / SEED_TEMPERATURE
    seed_weights = np.exp(np.clip(logits, -50.0, 0.0))
    seed_weights /= seed_weights.sum()
    particle = np.average(np.stack(paths), axis=0, weights=seed_weights)
    diagnostics = {
        'boundary': boundary,
        'suffix_rows': len(frame) - boundary,
        'gr_sigma': gr_sigma,
        'initial_rate': rate0,
        'effective_seeds': 1.0 / float(seed_weights @ seed_weights),
        'best_seed_weight': float(seed_weights.max()),
    }
    return boundary, particle, diagnostics

## Hidden-test inference

Seeds are assigned from the sorted test-well order, so a rerun is deterministic. The anchor prediction is the final finite prefix TVT repeated over the suffix. The deployed posterior mean is exactly `0.375 * anchor + 0.625 * particle`; there is no confidence gate or test-specific threshold.

In [ ]:
prediction_by_id = {}
diagnostics = []
started = time.perf_counter()

for number, path in enumerate(test_paths, start=1):
    well_id = path.name.removesuffix('__horizontal_well.csv')
    frame = pd.read_csv(path)
    typewell_path = path.with_name(f'{well_id}__typewell.csv')
    if not typewell_path.exists():
        raise FileNotFoundError(f'Missing paired typewell: {typewell_path.name}')
    typewell = pd.read_csv(typewell_path)
    boundary, particle, diag = particle_ensemble(
        frame, typewell, GLOBAL_SEED + number * 1000,
    )
    anchor_value = float(frame.loc[boundary - 1, 'TVT_input'])
    anchor = np.full(len(particle), anchor_value)
    blended = (1.0 - PARTICLE_WEIGHT) * anchor + PARTICLE_WEIGHT * particle
    if len(blended) != len(frame) - boundary or not np.isfinite(blended).all():
        raise AssertionError(f'{well_id}: invalid prediction vector')
    for row_index, value in zip(range(boundary, len(frame)), blended):
        prediction_by_id[f'{well_id}_{row_index}'] = float(value)
    diagnostics.append({'well_id': well_id, **diag})
    if number <= 5 or number % 25 == 0 or number == len(test_paths):
        print(
            f'{number}/{len(test_paths)} {well_id} rows={len(blended)} '
            f'GR_sigma={diag["gr_sigma"]:.2f} effective_seeds={diag["effective_seeds"]:.2f}'
        )

diagnostics = pd.DataFrame(diagnostics)
print('Inference seconds:', round(time.perf_counter() - started, 1))
display(diagnostics.describe(include='all'))

## Exact submission audit

Predictions are keyed by the official composite ID (`well_id` and original row index) and then mapped into the sample's exact order. The set equality assertion catches missing or unexpected suffix rows; the remaining assertions protect against schema drift, duplicate IDs, reordering, truncated output, and NaN/Inf values.

In [ ]:
sample = pd.read_csv(SAMPLE_PATH)
assert list(sample.columns) == ['id', 'tvt'], f'Unexpected sample columns: {list(sample.columns)}'
sample_ids = sample['id'].astype(str)
assert set(prediction_by_id) == set(sample_ids), 'Prediction IDs do not match sample IDs'

submission = pd.DataFrame({
    'id': sample_ids,
    'tvt': sample_ids.map(prediction_by_id).astype(float),
})
assert list(submission.columns) == ['id', 'tvt']
assert submission['id'].tolist() == sample_ids.tolist()
assert submission['id'].is_unique
assert len(submission) == len(sample)
assert np.isfinite(submission['tvt']).all()

output_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Rows:', len(submission))
print('TVT range:', float(submission['tvt'].min()), float(submission['tvt'].max()))
display(submission.head())

## Reproducibility and limitations

This notebook is deterministic, CPU-only, internet-off, and depends only on the competition files. It is intentionally conservative: the anchor blend reduces catastrophic GR branch selection but also attenuates correct large corrections. Local validation estimates generalization under the supplied train boundary; it cannot guarantee hidden-test or leaderboard performance. No further weight variants should be submitted merely to search the leaderboard—new submissions should require a new locally validated idea.